# Paper 3: Hur, Kim & Park (2022) — QCNN on Fashion-MNIST
### *"Quantum convolutional neural network for classical data classification"* (Quantum Machine Intelligence 4, 3 (2022) / arXiv:2108.00661)

---

## 1. Experimental Overview (Fashion-MNIST 0 vs 1)
This notebook extends the 8-qubit variational QCNN framework to **Fashion-MNIST** (Class 0: T-shirt/Top vs. Class 1: Trouser) [cite: 9].

### Key Technical Attributes:
1. **Qubit Allocation & Register Width ($N = 8$):**
   - Input image shape: $28 \times 28 = 784$ features [cite: 7].
   - Classical PCA reduces the 784 dimensions down to the top **8 principal components** [cite: 6, 8].
   - Hierarchical binary pooling ($8 \to 4 \to 2 \to 1$) over $L = 3$ layers requires an initial width of $2^3 = 8$ qubits to yield exactly 1 measured qubit [cite: 2].
2. **Encoding Benchmarks:**
   - **Qubit (Angle) Encoding:** $\mathcal{O}(1)$ depth parallel $R_y(x_i)$ rotations on 8 wires [cite: 4].
   - **Amplitude Encoding:** $L_2$-normalized statevector initialization [cite: 3].
3. **Ansatz Ablation:** Circuit 6 ($SO(4)$, 24 parameters) vs. Circuit 9 ($SU(4)$, 51 parameters) [cite: 5, 9].
4. **Measurement Strategies:** $P(|0\rangle), P(|1\rangle)$ (Cross-Entropy) vs. $\langle Z \rangle$ (MSE Loss) [cite: 5, 6].
5. **Optimization Suite:** Adam vs. SGD with Momentum (0.9) vs. RMSprop [cite: 7].

### Step 0: Environment Setup

In [ ]:
!pip install pennylane torch torchvision scikit-learn matplotlib scipy

### Step 1: Simulator Device & Precision Setup

In [ ]:
import os
import time
import pickle
import numpy as np
import matplotlib.pyplot as plt

import pennylane as qml
import torch
import torch.nn as nn
from torchvision import datasets
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

# Enforce Double Precision
torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Device: {device} | Precision: float64")

### Step 2: Fashion-MNIST Data Loading & PCA-8 Preprocessing
Filter Class 0 (T-shirt/Top) and Class 1 (Trouser). Fits PCA-8 on training images [cite: 6, 7].

In [ ]:
cache_dir = "./cache_fashion"
os.makedirs(cache_dir, exist_ok=True)
cache_file = os.path.join(cache_dir, "fashion_mnist_pca8_binary_0_1.npz")

if os.path.exists(cache_file):
    data = np.load(cache_file)
    X_tr_angle = data["X_tr_angle"]
    X_tr_amp = data["X_tr_amp"]
    y_train = data["y_train"]
    X_te_angle = data["X_te_angle"]
    X_te_amp = data["X_te_amp"]
    y_test = data["y_test"]
    print("Loaded preprocessed Fashion-MNIST PCA-8 dataset from cache.")
else:
    print("Downloading Fashion-MNIST and extracting 0 (T-shirt) vs 1 (Trouser)...")
    train_ds = datasets.FashionMNIST(root='./data_fashion', train=True, download=True)
    test_ds = datasets.FashionMNIST(root='./data_fashion', train=False, download=True)

    tr_mask = (train_ds.targets == 0) | (train_ds.targets == 1)
    te_mask = (test_ds.targets == 0) | (test_ds.targets == 1)

    X_tr_raw = train_ds.data[tr_mask].numpy().reshape(-1, 784).astype(np.float64) / 255.0
    y_tr_raw = train_ds.targets[tr_mask].numpy()
    X_te_raw = test_ds.data[te_mask].numpy().reshape(-1, 784).astype(np.float64) / 255.0
    y_te_raw = test_ds.targets[te_mask].numpy()

    y_train = (y_tr_raw == 1).astype(np.int64)
    y_test = (y_te_raw == 1).astype(np.int64)

    pca = PCA(n_components=8, random_state=42)
    X_tr_pca = pca.fit_transform(X_tr_raw)
    X_te_pca = pca.transform(X_te_raw)

    scaler = MinMaxScaler(feature_range=(0, np.pi))
    X_tr_angle = scaler.fit_transform(X_tr_pca)
    X_te_angle = scaler.transform(X_te_pca)

    def l2_norm(X):
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return X / norms

    X_tr_amp = l2_norm(X_tr_pca)
    X_te_amp = l2_norm(X_te_pca)

    np.savez(cache_file, X_tr_angle=X_tr_angle, X_tr_amp=X_tr_amp, y_train=y_train,
             X_te_angle=X_te_angle, X_te_amp=X_te_amp, y_test=y_test)
    print("Preprocessed and cached Fashion-MNIST PCA-8 dataset.")

print(f"Training set: {len(y_train)} samples | Test set: {len(y_test)} samples")
print(f"Angle range: [{X_tr_angle.min():.2f}, {X_tr_angle.max():.2f}] rad | Sample norm: {np.linalg.norm(X_tr_amp[0]):.4f}")

### Step 3: Quantum Encodings & Ansatze

In [ ]:
def qubit_encoding(x, wires):
    for i, w in enumerate(wires):
        qml.RY(x[i], wires=w)

def amplitude_encoding(x, wires):
    dim = 2 ** len(wires)
    padded = np.zeros(dim, dtype=np.float64)
    norm = np.linalg.norm(x)
    padded[:len(x)] = x / (norm if norm != 0 else 1.0)
    qml.StatePrep(padded, wires=wires)

def conv_ansatz_6(params, wires):
    qml.RY(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.RY(params[2], wires=wires[0])
    qml.RY(params[3], wires=wires[1])
    qml.CNOT(wires=[wires[1], wires[0]])
    qml.RY(params[4], wires=wires[0])
    qml.RY(params[5], wires=wires[1])

def conv_ansatz_9(params, wires):
    qml.RZ(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[0])
    qml.RZ(params[2], wires=wires[0])
    qml.RZ(params[3], wires=wires[1])
    qml.RY(params[4], wires=wires[1])
    qml.RZ(params[5], wires=wires[1])
    qml.CNOT(wires=[wires[1], wires[0]])
    qml.RZ(params[6], wires=wires[0])
    qml.RY(params[7], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.RY(params[8], wires=wires[1])
    qml.CNOT(wires=[wires[1], wires[0]])
    qml.RZ(params[9], wires=wires[0])
    qml.RY(params[10], wires=wires[0])
    qml.RZ(params[11], wires=wires[0])
    qml.RZ(params[12], wires=wires[1])
    qml.RY(params[13], wires=wires[1])
    qml.RZ(params[14], wires=wires[1])

def pool_ansatz(params, wires):
    qml.CRZ(params[0], wires=[wires[0], wires[1]])
    qml.PauliX(wires=wires[0])
    qml.CRX(params[1], wires=[wires[0], wires[1]])
    qml.PauliX(wires=wires[0])

def qcnn_conv_layer(ansatz_fn, params, wires):
    n = len(wires)
    for i in range(0, n - 1, 2):
        ansatz_fn(params, wires=[wires[i], wires[i + 1]])
    for i in range(1, n - 1, 2):
        ansatz_fn(params, wires=[wires[i], wires[i + 1]])
    if n > 2:
        ansatz_fn(params, wires=[wires[-1], wires[0]])

def qcnn_pool_layer(params, source_wires, sink_wires):
    for s, t in zip(source_wires, sink_wires):
        pool_ansatz(params, wires=[s, t])

### Step 4: 8-Qubit QNode and PyTorch Classifier

In [ ]:
dev8 = qml.device("default.qubit", wires=8)

def build_qcnn_qnode(ansatz_name="circuit_6", encoding_name="qubit", output_mode="probs"):
    ansatz_fn = conv_ansatz_6 if ansatz_name == "circuit_6" else conv_ansatz_9
    p_conv = 6 if ansatz_name == "circuit_6" else 15
    encode_fn = qubit_encoding if encoding_name == "qubit" else amplitude_encoding

    @qml.qnode(dev8, interface="torch", diff_method="backprop")
    def circuit(x, params):
        encode_fn(x, wires=range(8))
        c1 = params[0 : p_conv]
        p1 = params[p_conv : p_conv + 2]
        c2 = params[p_conv + 2 : 2 * p_conv + 2]
        p2 = params[2 * p_conv + 2 : 2 * p_conv + 4]
        c3 = params[2 * p_conv + 4 : 3 * p_conv + 4]
        p3 = params[3 * p_conv + 4 : 3 * p_conv + 6]

        # Layer 1: 8 -> 4
        qcnn_conv_layer(ansatz_fn, c1, wires=[0, 1, 2, 3, 4, 5, 6, 7])
        qcnn_pool_layer(p1, source_wires=[0, 2, 4, 6], sink_wires=[1, 3, 5, 7])
        # Layer 2: 4 -> 2
        qcnn_conv_layer(ansatz_fn, c2, wires=[1, 3, 5, 7])
        qcnn_pool_layer(p2, source_wires=[1, 5], sink_wires=[3, 7])
        # Layer 3: 2 -> 1
        ansatz_fn(c3, wires=[3, 7])
        pool_ansatz(p3, wires=[3, 7])

        if output_mode == "probs":
            return qml.probs(wires=7)
        else:
            return qml.expval(qml.PauliZ(7))
    return circuit

class ClassicalQCNNClassifier(nn.Module):
    def __init__(self, ansatz_name="circuit_6", encoding_name="qubit", output_mode="probs", seed=42):
        super().__init__()
        self.ansatz_name = ansatz_name
        self.encoding_name = encoding_name
        self.output_mode = output_mode

        p_conv = 6 if ansatz_name == "circuit_6" else 15
        num_params = 3 * p_conv + 6
        torch.manual_seed(seed)
        self.params = nn.Parameter(0.1 * torch.randn(num_params, dtype=torch.float64))
        self.qnode = build_qcnn_qnode(ansatz_name, encoding_name, output_mode)

    def forward(self, x_batch):
        if x_batch.ndim == 1:
            return self.qnode(x_batch, self.params)
        return torch.stack([self.qnode(x, self.params) for x in x_batch])

class TinyCNN(nn.Module):
    def __init__(self, out_channels=2):
        super().__init__()
        self.conv1 = nn.Conv1d(1, out_channels, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(out_channels * 4, 2)

    def forward(self, x):
        if x.ndim == 1:
            x = x.unsqueeze(0)
        x = x.unsqueeze(1)
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x)

### Step 5: Training Function

In [ ]:
def train_model(model, X_train, y_train, X_test, y_test, loss_type="ce",
                batch_size=25, n_iterations=200, lr=0.01, opt_name="Adam"):
    is_classical = isinstance(model, TinyCNN)
    if opt_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    elif opt_name == "SGD_Momentum":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_name == "RMSprop":
        optimizer = torch.optim.RMSprop(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    loss_history = []
    n_train = len(X_train)

    for it in range(n_iterations):
        idx = np.random.choice(n_train, batch_size, replace=False)
        batch_x = torch.tensor(X_train[idx], dtype=torch.float64)
        batch_y = torch.tensor(y_train[idx], dtype=torch.long)

        optimizer.zero_grad()
        if is_classical:
            logits = model(batch_x)
            if loss_type == "ce":
                loss = nn.CrossEntropyLoss()(logits, batch_y)
            else:
                y_targ = (1.0 - 2.0 * batch_y.to(torch.float64)).unsqueeze(1)
                pred_val = logits[:, 0:1] - logits[:, 1:2]
                loss = nn.MSELoss()(pred_val, y_targ)
        else:
            if loss_type == "ce":
                probs = model(batch_x)
                eps = 1e-9
                loss = -torch.mean(batch_y * torch.log(probs[:, 1] + eps) +
                                   (1.0 - batch_y) * torch.log(probs[:, 0] + eps))
            else:
                exp_vals = model(batch_x)
                y_targ = 1.0 - 2.0 * batch_y.to(torch.float64)
                loss = torch.mean((exp_vals - y_targ) ** 2)

        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())

    test_len = min(500, len(X_test))
    te_x = torch.tensor(X_test[:test_len], dtype=torch.float64)
    te_y = y_test[:test_len]

    model.eval()
    with torch.no_grad():
        if is_classical:
            preds = torch.argmax(model(te_x), dim=1).numpy()
        else:
            if loss_type == "ce":
                probs = model(te_x)
                preds = (probs[:, 1] > probs[:, 0]).long().numpy()
            else:
                preds = (model(te_x) < 0.0).long().numpy()

    return {"final_acc": np.mean(preds == te_y), "loss_history": np.array(loss_history)}

### Step 6: Measurement Technique Benchmark (Cross-Entropy vs MSE Loss)

In [ ]:
seeds = [42, 43, 44, 45, 46]
ce_accs, mse_accs = [], []
ce_losses, mse_losses = [], []

print("Evaluating Measurement Formulations on Fashion-MNIST (Circuit 6, Qubit Encoding, 5 Seeds)...")
for s in seeds:
    m_ce = ClassicalQCNNClassifier("circuit_6", "qubit", "probs", seed=s)
    res_ce = train_model(m_ce, X_tr_angle, y_train, X_te_angle, y_test, loss_type="ce")
    ce_accs.append(res_ce["final_acc"])
    ce_losses.append(res_ce["loss_history"])

    m_mse = ClassicalQCNNClassifier("circuit_6", "qubit", "expval", seed=s)
    res_mse = train_model(m_mse, X_tr_angle, y_train, X_te_angle, y_test, loss_type="mse")
    mse_accs.append(res_mse["final_acc"])
    mse_losses.append(res_mse["loss_history"])
    print(f"  Seed {s} -> CE: {res_ce['final_acc']*100:.2f}% | MSE: {res_mse['final_acc']*100:.2f}%")

print(f"\nCross-Entropy (Probabilities) : {np.mean(ce_accs)*100:.2f}% ± {np.std(ce_accs)*100:.2f}%")
print(f"Mean Squared Error (Pauli-Z)   : {np.mean(mse_accs)*100:.2f}% ± {np.std(mse_accs)*100:.2f}%")

### Step 7: Optimizer Benchmark (Adam vs. SGD-Momentum vs. RMSprop)

In [ ]:
optimizers_list = ["Adam", "SGD_Momentum", "RMSprop"]
opt_benchmark_results = {}
opt_seeds = [42, 43, 44]

print("Running Optimizer Benchmark on Fashion-MNIST (Circuit 6, Qubit Encoding, 150 iterations)...")
for opt in optimizers_list:
    opt_accs = []
    for s in opt_seeds:
        m = ClassicalQCNNClassifier("circuit_6", "qubit", "probs", seed=s)
        res = train_model(m, X_tr_angle, y_train, X_te_angle, y_test, loss_type="ce",
                          n_iterations=150, lr=0.01, opt_name=opt)
        opt_accs.append(res["final_acc"])
    mean_acc = np.mean(opt_accs) * 100
    std_acc = np.std(opt_accs) * 100
    opt_benchmark_results[opt] = (mean_acc, std_acc)
    print(f"  Optimizer: {opt:<14} | Acc: {mean_acc:.2f}% ± {std_acc:.2f}%")

### Step 8: Full Benchmark Grid & Table II Reproduction

In [ ]:
NUM_SEEDS = 3
GRID_ITERS = 200
ENCODINGS = ["qubit", "amplitude"]
ansatze = ["circuit_6", "circuit_9"]
SEEDS = seeds[:NUM_SEEDS]

CACHE = "bench_cache_fashion.pkl"
cache = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) else {}

def run_cfg(key, make_model, X_tr, X_te, seed):
    key = key + (GRID_ITERS,)
    if key in cache:
        return cache[key]
    np.random.seed(seed); torch.manual_seed(seed)
    res = train_model(make_model(), X_tr, y_train, X_te, y_test,
                      loss_type="ce", n_iterations=GRID_ITERS, opt_name="Adam")
    cache[key] = {"final_acc": res["final_acc"], "loss_history": res["loss_history"]}
    pickle.dump(cache, open(CACHE, "wb"))
    return cache[key]

all_enc = ["qubit", "amplitude", "dense"]
table_2_results = {a: {e: [] for e in all_enc} for a in ansatze}
all_loss_curves = {a: {e: [] for e in all_enc} for a in ansatze}

skip = set()
if GRID_ITERS == len(ce_losses[0]):
    table_2_results["circuit_6"]["qubit"] = list(ce_accs[:NUM_SEEDS])
    all_loss_curves["circuit_6"]["qubit"] = list(ce_losses[:NUM_SEEDS])
    skip = {("circuit_6", "qubit")}

for a in ansatze:
    for e in ENCODINGS:
        if (a, e) in skip:
            continue
        X_tr = X_tr_amp if e == "amplitude" else X_tr_angle
        X_te = X_te_amp if e == "amplitude" else X_te_angle
        for s in SEEDS:
            t0 = time.time()
            r = run_cfg((a, e, s),
                        lambda: ClassicalQCNNClassifier(a, e, "probs", seed=s),
                        X_tr, X_te, s)
            table_2_results[a][e].append(r["final_acc"])
            all_loss_curves[a][e].append(r["loss_history"])
            print(f"{a} | {e} | seed {s} | acc {r['final_acc']*100:.2f}% | elapsed {time.time()-t0:.0f}s", flush=True)

tiny26_accs, tiny26_losses, tiny50_accs, tiny50_losses = [], [], [], []
for s in SEEDS:
    for oc, accs, losses in [(2, tiny26_accs, tiny26_losses), (4, tiny50_accs, tiny50_losses)]:
        np.random.seed(s); torch.manual_seed(s)
        r = train_model(TinyCNN(out_channels=oc), X_tr_angle, y_train, X_te_angle, y_test,
                        loss_type="ce", n_iterations=GRID_ITERS, opt_name="Adam")
        accs.append(r["final_acc"]); losses.append(r["loss_history"])
print("Fashion-MNIST benchmark grid completed.")

### Step 9: Formatted Table II & Performance Visualizations

In [ ]:
print("=" * 95)
print(f"TABLE II: FASHION-MNIST (0 vs 1) REPLICATION (HUR ET AL. 2022 | {NUM_SEEDS} SEEDS)")
print(f"{'Architecture / Ansatz':<28} | {'Params':<8} | {'Qubit (Angle)':<16} | {'Amplitude':<16} | {'Dense Qubit':<16}")
print("-" * 95)

for a, name, pcount in [("circuit_6", "QCNN Circuit 6 (SO(4))", 24), ("circuit_9", "QCNN Circuit 9 (SU(4))", 51)]:
    q_str = f"{np.mean(table_2_results[a]['qubit'])*100:.2f} ± {np.std(table_2_results[a]['qubit'])*100:.2f}%"
    a_str = f"{np.mean(table_2_results[a]['amplitude'])*100:.2f} ± {np.std(table_2_results[a]['amplitude'])*100:.2f}%"
    d_str = "N/A (Omitted)"
    print(f"{name:<28} | {pcount:<8} | {q_str:<16} | {a_str:<16} | {d_str:<16}")

t26_str = f"{np.mean(tiny26_accs)*100:.2f} ± {np.std(tiny26_accs)*100:.2f}%"
t50_str = f"{np.mean(tiny50_accs)*100:.2f} ± {np.std(tiny50_accs)*100:.2f}%"
print("-" * 95)
print(f"{'Classical TinyCNN-26':<28} | {26:<8} | {t26_str:<16} | {'N/A (Classical)':<16} | {'N/A (Classical)':<16}")
print(f"{'Classical TinyCNN-50':<28} | {50:<8} | {t50_str:<16} | {'N/A (Classical)':<16} | {'N/A (Classical)':<16}")
print("=" * 95)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
iters = np.arange(1, 201)

# Panel 1: Loss comparison
axes[0].plot(iters, np.mean(ce_losses, axis=0), label=f"Cross-Entropy ({np.mean(ce_accs)*100:.1f}%)", color="#1f77b4", lw=2)
axes[0].plot(iters, np.mean(mse_losses, axis=0), label=f"MSE Loss ({np.mean(mse_accs)*100:.1f}%)", color="#d62728", lw=2)
axes[0].set_title("Measurement Technique Comparison\n(Fashion-MNIST C6, 5 Seeds)", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Iteration", fontsize=10); axes[0].set_ylabel("Loss", fontsize=10)
axes[0].grid(True, linestyle="--", alpha=0.5); axes[0].legend(fontsize=9)

# Panel 2: Model convergence
axes[1].plot(iters, np.mean(all_loss_curves["circuit_6"]["qubit"], axis=0), label="QCNN C6 (24 p)", color="#2ca02c", lw=2)
axes[1].plot(iters, np.mean(all_loss_curves["circuit_9"]["qubit"], axis=0), label="QCNN C9 (51 p)", color="#9467bd", lw=2)
axes[1].plot(iters, np.mean(tiny26_losses, axis=0), label="TinyCNN (26 p)", color="#ff7f0e", lw=2, linestyle="--")
axes[1].set_title("Convergence Dynamics\n(Fashion-MNIST Qubit Encoding)", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Iteration", fontsize=10); axes[1].set_ylabel("Loss", fontsize=10)
axes[1].grid(True, linestyle="--", alpha=0.5); axes[1].legend(fontsize=9)

# Panel 3: Test Accuracies
models_labels = ["QCNN C6\n(Angle)", "QCNN C6\n(Amp)", "QCNN C9\n(Angle)", "QCNN C9\n(Amp)", "TinyCNN\n(26 p)", "TinyCNN\n(50 p)"]
means = [np.mean(table_2_results["circuit_6"]["qubit"])*100, np.mean(table_2_results["circuit_6"]["amplitude"])*100,
         np.mean(table_2_results["circuit_9"]["qubit"])*100, np.mean(table_2_results["circuit_9"]["amplitude"])*100,
         np.mean(tiny26_accs)*100, np.mean(tiny50_accs)*100]
stds = [np.std(table_2_results["circuit_6"]["qubit"])*100, np.std(table_2_results["circuit_6"]["amplitude"])*100,
        np.std(table_2_results["circuit_9"]["qubit"])*100, np.std(table_2_results["circuit_9"]["amplitude"])*100,
        np.std(tiny26_accs)*100, np.std(tiny50_accs)*100]
axes[2].bar(models_labels, means, yerr=stds, capsize=4, color=["#2ca02c", "#66c2a5", "#9467bd", "#8da0cb", "#ff7f0e", "#fc8d62"], alpha=0.85, edgecolor="black")
axes[2].set_ylim(min(means) - 10, 102)
axes[2].set_title(f"Test Accuracy Comparison\n(Mean ± Std over {NUM_SEEDS} Seeds)", fontsize=11, fontweight="bold")
axes[2].set_ylabel("Test Accuracy (%)", fontsize=10)
axes[2].grid(True, axis="y", linestyle="--", alpha=0.5)

plt.suptitle("QCNN Benchmark on Fashion-MNIST (0 vs 1)", fontsize=13, y=1.02, fontweight="bold")
plt.tight_layout()
plt.show()